In [1]:
import os
import pathlib
import random

import kagglehub
import matplotlib.pyplot as plt
import tensorflow as tf

print(tf.__version__)

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
# Download the Tom and Jerry dataset from Kaggle
path = kagglehub.dataset_download("balabaskar/tom-and-jerry-image-classification")
print("Path to dataset files:", path)

# Look at the folder structure
for root, dirs, files in os.walk(path):
    print(root)
    # only show first few items for brevity
    print("  subfolders:", dirs[:5])
    print("  files:", files[:5])
    break

In [ ]:
# Assume images are in a subfolder structure like:
# path/
#   Tom/
#   Jerry/
# If the structure is different, print above cell output and adjust this.

data_dir = pathlib.Path(path)
image_count = len(list(data_dir.rglob("*.jpg"))) + len(list(data_dir.rglob("*.png")))
print("Total images found:", image_count)

# List class names (folder names) at first level
class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
print("Classes:", class_names)

In [ ]:
# Show a few random example images
image_paths = list(data_dir.rglob("*.jpg")) + list(data_dir.rglob("*.png"))
random.shuffle(image_paths)

plt.figure(figsize=(8, 8))
for i, img_path in enumerate(image_paths[:9]):
    img = tf.keras.utils.load_img(img_path)
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(img_path.parent.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Create TensorFlow datasets from the image folders
img_size = (128, 128)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Class names:", class_names)

In [ ]:
# Prefetch for better performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# Build a simple CNN model in TensorFlow
num_classes = len(class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=img_size + (3,)),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
# Plot training and validation accuracy
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure()
plt.plot(acc, label="train acc")
plt.plot(val_acc, label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()